In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace = True)

In [7]:
df.shape

(49582, 2)

In [8]:
# Pre processing 
df["review"]= df["review"].str.lower()

In [9]:
import re

In [10]:
def remove_urls(text):
    text = re.sub(r"http\S+","",text)
    return text

df["review"]=df["review"].apply(remove_urls)

In [11]:
#Remove punctuations

def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text 

df["review"] = df["review"].apply(remove_punctuations)

In [12]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [13]:
def remove_html(text):
    text = re.sub(r"<.*?>","", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [14]:
# Removing the stopwords

import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

c:\Users\aniru\anaconda3\envs\qgss2024\Lib\site-packages\nltk\metrics\association.py:26: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.5.1)
  from scipy.stats import fisher_exact
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aniru\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aniru\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aniru\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


In [16]:
sample_text = "I like coding in python!"
tokens = word_tokenize(sample_text)

In [17]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [18]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [19]:
# Stemming 

from nltk.stem import PorterStemmer

In [20]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [21]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [22]:
#Encoding 
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [23]:
y = df["sentiment"]

In [24]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [25]:
# Vectorization 
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(df["review"])

In [27]:
# Dataset and Data Loaders

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size = 0.2, random_state = 42
)

In [28]:
X_train.shape

(39665, 5000)

In [31]:
type(X_train)

numpy.ndarray

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader


In [30]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [32]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [33]:
train_loader = DataLoader(train_set, shuffle = True, batch_size = 64)
test_loader = DataLoader(test_set, shuffle = True, batch_size = 64)


In [34]:
# Building the RNN

import torch.nn as nn
import torch.optim as optim


In [36]:
class RNN(nn.Module):
    def __init__(self, input_size,hidden_size = 128, num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size , num_layers , batch_first = True)

        self.fc = nn.Linear(hidden_size , 1)


    def forward(self,x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hiddens_size)
        out , _ = self.rnn(x,h0)

        out = self.fc(out[:,-1,:])
        return out
    



In [37]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())
